# Module 5: Data Cleaning in Pandas
### 🚢 Using the Kaggle Titanic Dataset

## 1️⃣ Introduction

In real-world datasets, the data is **rarely perfect**.
Most datasets contain problems such as:

- Missing values
- Duplicate records
- Incorrect data
- Inconsistent values

This process of fixing data problems is called **Data Cleaning**.

Data cleaning is one of the **most important tasks in data analysis**, because incorrect or incomplete data can lead to **wrong analysis results**.

In this module we will learn:

- `isnull()`
- `notnull()`
- `dropna()`
- `fillna()`
- `replace()`
- `drop_duplicates()`

We will practise all these operations on the **real-world Kaggle Titanic dataset**.

## 2️⃣ Why This Concept is Important

The Titanic dataset is a perfect example of **real-world messy data**:

| Column   | Problem                                      |
| -------- | -------------------------------------------- |
| `Age`    | ~20% values are missing                      |
| `Cabin`  | ~77% values are missing                      |
| `Embarked` | 2 missing values                           |
| `Fare`   | Some zero values that may be incorrect       |

If we skip cleaning and go straight to analysis, our results will be **wrong or misleading**.

Using Pandas data cleaning tools we can fix all of these issues.

## 3️⃣ Load the Titanic Dataset

We load the dataset directly from a public URL — no file download needed.

In [ ]:
import pandas as pd
import numpy as np

# Load the Titanic dataset

df = pd.read_csv("Titanic-Dataset.csv")

print("Shape:", df.shape)
df.head()

## 4️⃣ `isnull()` Function

### Purpose

Detects **missing values** in the dataset.

Returns a DataFrame of booleans:
- **True → Missing value (NaN)**
- **False → Value is present**

In [ ]:
# Detect missing values across the entire Titanic DataFrame
df.isnull().head(10)

### Count Missing Values Per Column

This is the **first thing a data analyst does** after loading a dataset.

In [ ]:
# Count total missing values in each column
missing = df.isnull().sum()
print(missing)
print("\nTotal missing values:", missing.sum())

In [ ]:
# Calculate missing percentage per column
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct.sort_values(ascending=False))

## 5️⃣ `notnull()` Function

### Purpose

The opposite of `isnull()` — checks which values are **not missing**.

- **True → value is present**
- **False → value is missing**

Useful for filtering rows where a column has valid data.

In [ ]:
# Check which Age values are NOT missing
df["Age"].notnull().head(10)

In [ ]:
# Use notnull() to filter — show only rows where Age is present
df[df["Age"].notnull()][["Name", "Age", "Survived"]].head()

## 6️⃣ `dropna()` Function

### Purpose

Removes rows (or columns) that contain **missing values**.

Use this when incomplete rows are not useful for analysis.

In [ ]:
# Drop ALL rows that have any missing value
df_dropped = df.dropna()
print("Original shape:", df.shape)
print("After dropna() shape:", df_dropped.shape)

In [ ]:
# Drop rows where ONLY the 'Age' column is missing
df_age_clean = df.dropna(subset=["Age"])
print("Rows after dropping missing Age:", df_age_clean.shape[0])
print("Missing Age values remaining:", df_age_clean["Age"].isnull().sum())

### Remove Columns with Missing Values

Use `axis=1` to drop **columns** with missing values instead of rows.

```
axis=0 → rows (default)
axis=1 → columns
```

In [ ]:
# Drop columns that contain ANY missing value
df_col_dropped = df.dropna(axis=1)
print("Original columns:", df.columns.tolist())
print("After dropna(axis=1):", df_col_dropped.columns.tolist())

## 7️⃣ `fillna()` Function

### Purpose

**Replaces missing values** with a specified value instead of dropping them.

Use this when the row is valuable and you don't want to lose it.

In [ ]:
# Fill ALL missing values with 0 (to see the effect)
df.fillna(0).isnull().sum()

### Fill `Age` with Mean Value

For numerical columns, filling with the **mean** is the most common and statistically sound approach.

In [ ]:
# Check mean age before filling
mean_age = df["Age"].mean()
print(f"Mean Age: {mean_age:.2f}")
print(f"Missing Age before: {df['Age'].isnull().sum()}")

# Fill missing Age with mean age
df["Age"] = df["Age"].fillna(mean_age)

print(f"Missing Age after:  {df['Age'].isnull().sum()}")

### Fill `Embarked` with Mode Value

For **categorical columns**, filling with the **mode** (most frequent value) is the best practice.

In [ ]:
# Check Embarked column
print("Missing Embarked:", df["Embarked"].isnull().sum())
print("Value counts:\n", df["Embarked"].value_counts())

# Fill missing Embarked with mode (most frequent port)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print("\nMissing Embarked after fill:", df["Embarked"].isnull().sum())

### Fill `Cabin` with a Placeholder String

Since `Cabin` has ~77% missing values, we fill it with `'Unknown'` rather than dropping the column entirely.

In [ ]:
# Fill missing Cabin values with 'Unknown'
df["Cabin"] = df["Cabin"].fillna("Unknown")

print("Missing Cabin after fill:", df["Cabin"].isnull().sum())
df[["Name", "Cabin"]].head()

## 8️⃣ `replace()` Function

### Purpose

Replace **specific values** in the dataset with new values.

Useful for correcting inconsistent entries or making values more readable.

In [ ]:
# Replace numeric Survived values with readable labels
df["Survived"].value_counts()

In [ ]:
# Replace 0 → 'No' and 1 → 'Yes' in the Survived column
df["Survived"] = df["Survived"].replace({0: "No", 1: "Yes"})

print(df[["Name", "Survived"]].head(10))

In [ ]:
# Replace Sex values: 'male' → 'Male' and 'female' → 'Female'
df["Sex"] = df["Sex"].replace({"male": "Male", "female": "Female"})

print(df["Sex"].value_counts())

## 9️⃣ `drop_duplicates()` Function

### Purpose

Removes **duplicate rows** from the DataFrame.

Duplicate records can distort analysis results (e.g. inflated counts, wrong averages).

In [ ]:
# Check for duplicates in the original Titanic dataset
print("Total duplicate rows:", df.duplicated().sum())

In [ ]:
# Simulate duplicates by appending a few rows again
df_with_dupes = pd.concat([df, df.head(5)], ignore_index=True)
print("Shape with duplicates:", df_with_dupes.shape)
print("Duplicate rows:", df_with_dupes.duplicated().sum())

In [ ]:
# Remove duplicate rows
df_clean = df_with_dupes.drop_duplicates()
print("Shape after drop_duplicates():", df_clean.shape)
print("Duplicate rows remaining:", df_clean.duplicated().sum())

## 🌍 Real-World Example — Full Cleaning Workflow

Combining all cleaning steps into one complete, production-ready pipeline.

In [ ]:
import pandas as pd

# Step 1: Load a fresh copy
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
print("--- Raw Dataset ---")
print("Shape:", df.shape)
print(df.isnull().sum())

In [ ]:
# Step 2: Fill missing numerical values with mean
df["Age"] = df["Age"].fillna(df["Age"].mean())
df["Fare"] = df["Fare"].fillna(df["Fare"].mean())

# Step 3: Fill missing categorical values
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Cabin"] = df["Cabin"].fillna("Unknown")

# Step 4: Replace values for readability
df["Survived"] = df["Survived"].replace({0: "No", 1: "Yes"})
df["Sex"] = df["Sex"].replace({"male": "Male", "female": "Female"})

# Step 5: Remove duplicates
df = df.drop_duplicates()

print("\n--- Cleaned Dataset ---")
print("Shape:", df.shape)
print(df.isnull().sum())

In [ ]:
# Preview the final cleaned dataset
df.head(10)

## 💡 Important Tips

✔ **Always check missing values first** before any analysis:
```python
df.isnull().sum()
```

✔ For **numerical columns** → fill missing with `mean()`  
✔ For **categorical columns** → fill missing with `mode()` or a placeholder like `'Unknown'`  
✔ Use **`dropna(subset=["col"])`** to drop rows missing only in a specific column  
✔ Always **remove duplicates** before analysis to avoid skewed results

## 📋 Summary of Module 5

In this module we learned how to **clean data using Pandas** with the real-world **Titanic dataset**.

| Function            | Titanic Use Case                                      |
| ------------------- | ----------------------------------------------------- |
| `isnull()`          | Found missing Age, Cabin, Embarked values             |
| `notnull()`         | Filtered rows where Age is present                    |
| `dropna()`          | Dropped rows/columns with missing values              |
| `fillna()`          | Filled Age with mean, Embarked with mode, Cabin with 'Unknown' |
| `replace()`         | Replaced 0/1 Survived with 'No'/'Yes', fixed Sex labels |
| `drop_duplicates()` | Removed simulated duplicate passenger records         |

Data cleaning is one of the **most important steps in data analysis** — clean data leads to accurate results. 🚢🚀